# 🗳️ Thai Election OCR Pipeline
Extract structured voting data from scanned Thai election result documents (Form สส.6/1) from the 2026 Thai general election.

## 1️⃣ Imports
โหลด library ที่ใช้สำหรับจัดการไฟล์ภาพ, ทำ OCR, ประมวลผลข้อมูล, และบันทึกผลลัพธ์


In [1]:
!pip install requests pandas

In [2]:
from __future__ import annotations

import os
import json
import csv
import re
import base64
import random
import time
from pathlib import Path
from collections import defaultdict

import requests
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2️⃣ Configuration
กำหนดค่าตั้งต้นของระบบ เช่น API key, model ที่จะใช้, path ของ input/output


In [24]:
# ===== CONFIG =====
OPENROUTER_API_KEY = ""
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL_OCR = "google/gemini-3-flash-preview"
MODEL_VOTE_EXTRACTION = "google/gemini-3.1-flash-lite-preview"

INPUT_DIR = Path("/content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/images")
TEMPLATE_CSV = Path("/content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/submission_template_v3.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data")

MAX_RETRIES = 5
REQUEST_TIMEOUT = 180
REQUEST_DELAY = 0.0
USE_VOTE_EXTRACTION = True
USE_TOTAL_VERIFICATION = True
ZERO_VOTE_RATIO_THRESHOLD = 0.4

RAW_DIR = OUTPUT_DIR / "raw_outputs"
MERGED_OCR_DIR = OUTPUT_DIR / "merged_ocr"
VOTE_EXTRACTION_DIR = OUTPUT_DIR / "vote_extraction"
PARSED_DIR = OUTPUT_DIR / "parsed_outputs"
VALIDATION_DIR = OUTPUT_DIR / "validation"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"
SUBMISSION_FILLED_PATH = OUTPUT_DIR / "submission_filled.csv"
SUBMISSION_FINAL_PATH = OUTPUT_DIR / "submission_final.csv"

for d in [OUTPUT_DIR, RAW_DIR, MERGED_OCR_DIR, VOTE_EXTRACTION_DIR, PARSED_DIR, VALIDATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR =", INPUT_DIR)
print("TEMPLATE_CSV =", TEMPLATE_CSV)
print("OUTPUT_DIR =", OUTPUT_DIR)

INPUT_DIR = /content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/images
TEMPLATE_CSV = /content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/submission_template_v3.csv
OUTPUT_DIR = /content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data


## 3️⃣ Normalization Helpers
ใช้แปลงเลขไทยเป็นเลขอารบิก และลบอักขระที่ไม่จำเป็น เพื่อให้คะแนนและหมายเลขแถวอยู่ในรูปแบบมาตรฐานพร้อมนำไป map ต่อได้


In [6]:
THAI_TO_ARABIC = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
PAGE_SUFFIX_RE = re.compile(r"_page(?P<page>\d+)$")
ID_RE = re.compile(r"^(?P<doc_id>.+)_(?P<row_num>\d+)$")
PARTY_ALIASES = {"วิชชั่นใหม่": "วิชั่นใหม่"}

def thai_digits_to_arabic(text: str) -> str:
    return str(text or "").translate(THAI_TO_ARABIC)

def normalize_vote(text: str) -> str:
    text = thai_digits_to_arabic(text)
    text = re.sub(r"\([^)]*\)", "", text)
    text = text.replace(",", "")
    text = re.sub(r"[^0-9]", "", text)
    return text or "0"

def normalize_row_num(text: str) -> str:
    text = thai_digits_to_arabic(text)
    return re.sub(r"[^0-9]", "", text)

def normalize_party_name(text: str) -> str:
    text = thai_digits_to_arabic(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\"'.,;:()\[\]{}]", "", text)
    return PARTY_ALIASES.get(text, text)

def get_doc_id_from_image(path: Path) -> str:
    return PAGE_SUFFIX_RE.sub("", path.stem)

def page_sort_key(path: Path):
    m = PAGE_SUFFIX_RE.search(path.stem)
    page_number = int(m.group("page")) if m else 1
    return page_number, path.name

def infer_doc_type(doc_id: str) -> str:
    if doc_id.startswith("constituency_"):
        return "constituency"
    if doc_id.startswith("party_list_"):
        return "party_list"
    return "unknown"

def build_document_groups(images_dir: Path):
    groups = defaultdict(list)
    for p in sorted(images_dir.glob("*.png")):
        groups[get_doc_id_from_image(p)].append(p)
    return {doc_id: sorted(paths, key=page_sort_key) for doc_id, paths in sorted(groups.items())}

def split_template_id(row_id: str):
    m = ID_RE.match(row_id)
    if not m:
        raise ValueError(f"Cannot parse template id: {row_id}")
    return m.group("doc_id"), m.group("row_num")

def image_to_data_url(image_path: Path) -> str:
    suffix = image_path.suffix.lower()
    mime = "image/png" if suffix == ".png" else "image/jpeg"
    encoded = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{encoded}"

def extract_json(text: str):
    text = text.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            raise
        return json.loads(m.group(0))

def write_json(path: Path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

## 4️⃣ Load Template and Group Documents
โหลดไฟล์ submission_template_v3.csv และจัดกลุ่มภาพตาม doc_id เพื่อรวมหน้าที่อยู่ในเอกสารเดียวกันก่อนนำไปประมวลผลต่อ

In [17]:
import json
from pathlib import Path
import csv
from collections import defaultdict

def load_template(path: Path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        raw_rows = list(reader)

    header_map = {name.strip().lower().replace(" ", "_"): name for name in fieldnames}
    id_column = header_map["id"]
    votes_column = header_map["votes"]
    party_column = header_map.get("party_name")

    by_doc = defaultdict(list)
    for raw in raw_rows:
        row_id = str(raw.get(id_column, "")).strip()
        if not row_id:
            continue

        doc_id, row_num = split_template_id(row_id)

        by_doc[doc_id].append({
            "id": row_id,
            "doc_id": doc_id,
            "row_num": normalize_row_num(row_num),
            "party_name": normalize_party_name(str(raw.get(party_column, ""))) if party_column else "",
            "votes": normalize_vote(str(raw.get(votes_column, "0"))),
            "original": dict(raw),
        })

    return {
        "fieldnames": fieldnames,
        "raw_rows": raw_rows,
        "id_column": id_column,
        "votes_column": votes_column,
        "party_column": party_column,
        "by_doc": dict(by_doc),
    }

def default_checkpoint():
    return {
        "done_docs": [],
        "results": {},
        "strategy_used": {}
    }

def load_checkpoint(path: Path):
    if not path.exists():
        return default_checkpoint()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    ckpt = default_checkpoint()
    ckpt.update(data)
    return ckpt

def save_checkpoint(path: Path, checkpoint: dict):
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
    tmp_path.replace(path)

template = load_template(TEMPLATE_CSV)
doc_groups = build_document_groups(INPUT_DIR)

## 5️⃣ OpenRouter API functions
ฟังก์ชันสำหรับเรียกโมเดลผ่าน OpenRouter เพื่อทำ OCR จากภาพเอกสาร และทำ Vote Extraction จากผล OCR ที่รวมแล้วในระดับ document

In [18]:
def call_openrouter(model, messages, purpose="api"):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": messages,
        "response_format": {"type": "json_object"},
        "temperature": 0,
    }

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)

            if r.status_code == 429:
                wait = min(10 * (2 ** attempt), 60)
                print(f"rate limited during {purpose}, wait {wait}s")
                time.sleep(wait)
                continue

            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"]
            return {"ok": True, "data": extract_json(content)}

        except Exception as e:
            wait = min(10 * (2 ** attempt), 60)
            print(f"error during {purpose}: {e}")
            time.sleep(wait)

    return {"ok": False, "data": {"rows": []}}


def ocr_page(image_path, doc_id):
    doc_type = infer_doc_type(doc_id)

    if doc_type == "constituency":
        prompt = """อ่านข้อความจากภาพผลเลือกตั้ง สส.เขต แล้วคืน JSON เท่านั้น
{
  "page_type": "score_page" | "candidate_party_page" | "unknown",
  "declared_total_raw": "คะแนนรวมถ้ามี",
  "rows": [
    {"row_num":"หมายเลข", "candidate_no":"หมายเลขถ้าเห็น", "party_name":"ชื่อพรรคถ้าเห็น", "vote_raw":"คะแนนถ้าเห็น"}
  ]
}"""
    else:
        prompt = """อ่านข้อความจากภาพผลเลือกตั้งบัญชีรายชื่อ แล้วคืน JSON เท่านั้น
{
  "page_type": "score_page" | "unknown",
  "declared_total_raw": "คะแนนรวมถ้ามี",
  "rows": [
    {"row_num":"หมายเลข", "party_name":"ชื่อพรรคถ้าเห็น", "vote_raw":"คะแนนถ้าเห็น"}
  ]
}"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": f"doc_id={doc_id}\n{prompt}"},
            {"type": "image_url", "image_url": {"url": image_to_data_url(image_path)}},
        ],
    }]

    return call_openrouter(MODEL_OCR, messages, purpose=f"OCR {image_path.name}")


def vote_extraction(doc_id, merged_ocr, expected_rows, rule_rows):
    prompt = json.dumps({
        "instruction": 'Return JSON only: {"rows":[{"id":"expected_id","votes":"digit-only-or-0"}]}',
        "doc_id": doc_id,
        "expected_rows": expected_rows,
        "merged_ocr": merged_ocr,
        "rule_rows": rule_rows,
    }, ensure_ascii=False)

    messages = [{"role": "user", "content": prompt}]
    return call_openrouter(MODEL_VOTE_EXTRACTION, messages, purpose=f"vote extraction {doc_id}")

## 6️⃣ mapping / validation functions
ฟังก์ชันสำหรับ map ผล OCR เข้ากับ id ใน template และตรวจสอบความถูกต้องเบื้องต้นก่อนสร้างผลลัพธ์ final

In [19]:
def build_rule_rows(doc_id, expected_rows, ocr_rows):
    # map row_num -> row
    row_map = {}
    party_map = {}

    for row in ocr_rows:
        row_num = normalize_row_num(row.get("row_num", ""))
        party_name = normalize_party_name(row.get("party_name", ""))
        vote = normalize_vote(row.get("vote_raw", ""))

        if row_num and row_num not in row_map and vote != "0":
            row_map[row_num] = vote

        if party_name and party_name not in party_map and vote != "0":
            party_map[party_name] = vote

    final_rows = []
    for t in expected_rows:
        vote = "0"

        # ใช้ party_name ก่อน
        if t["party_name"] and t["party_name"] in party_map:
            vote = party_map[t["party_name"]]

        # fallback ไป row_num
        elif t["row_num"] in row_map:
            vote = row_map[t["row_num"]]

        final_rows.append({
            "id": t["id"],
            "doc_id": doc_id,
            "row_num": t["row_num"],
            "party_name": t["party_name"],
            "votes": vote,
        })

    return final_rows


def build_extraction_rows(doc_id, expected_rows, extracted_rows):
    extracted_map = {}
    expected_ids = {x["id"] for x in expected_rows}

    for row in extracted_rows:
        row_id = str(row.get("id", "")).strip()
        vote = normalize_vote(str(row.get("votes", "")))
        if row_id in expected_ids:
            extracted_map[row_id] = vote

    final_rows = []
    for t in expected_rows:
        final_rows.append({
            "id": t["id"],
            "doc_id": doc_id,
            "row_num": t["row_num"],
            "party_name": t["party_name"],
            "votes": extracted_map.get(t["id"], "0"),
        })

    return final_rows


def choose_final_rows(rule_rows, extraction_rows):
    # ใช้ extraction เติมเฉพาะจุดที่ rule เป็น 0
    if not extraction_rows:
        return rule_rows, "rule_based_only"

    extraction_map = {r["id"]: r["votes"] for r in extraction_rows}
    final_rows = []

    used_extraction = 0
    for row in rule_rows:
        new_vote = extraction_map.get(row["id"], "0")
        if row["votes"] == "0" and new_vote != "0":
            new_row = dict(row)
            new_row["votes"] = new_vote
            final_rows.append(new_row)
            used_extraction += 1
        else:
            final_rows.append(row)

    strategy = "hybrid" if used_extraction > 0 else "rule_based_only"
    return final_rows, strategy

## 7️⃣ Process Document and Run Pipeline
รวมทุกขั้นตอนของ pipeline ตั้งแต่ OCR ทีละหน้า, รวมผลระดับ document, ทำ mapping คะแนน, และ export ไฟล์ submission สำหรับส่งแข่งขัน

In [20]:
def process_document(doc_id, pages, expected_rows):
    print("=" * 80)
    print(f"[doc] {doc_id} pages={len(pages)}")

    page_outputs = []
    ocr_rows = []

    # OCR ทุกหน้า
    for page_idx, image_path in enumerate(pages, 1):
        print(f"page {page_idx}/{len(pages)} -> {image_path.name}")

        result = ocr_page(image_path, doc_id)
        write_json(RAW_DIR / f"{image_path.stem}.json", result)

        rows = result.get("data", {}).get("rows", [])
        print("rows extracted:", len(rows))

        for row in rows:
            row["source_page"] = image_path.name
            ocr_rows.append(row)

        page_outputs.append({
            "page": image_path.name,
            "rows": rows,
            "declared_total_raw": result.get("data", {}).get("declared_total_raw", ""),
            "page_type": result.get("data", {}).get("page_type", "unknown"),
        })

    merged_ocr = {
        "doc_id": doc_id,
        "pages": [p.name for p in pages],
        "page_outputs": page_outputs,
        "ocr_rows": ocr_rows,
    }
    write_json(MERGED_OCR_DIR / f"{doc_id}.json", merged_ocr)

    # rule-based
    rule_rows = build_rule_rows(doc_id, expected_rows, ocr_rows)

    # vote extraction
    extraction_rows = None
    if USE_VOTE_EXTRACTION:
        result = vote_extraction(doc_id, merged_ocr, expected_rows, rule_rows)
        write_json(VOTE_EXTRACTION_DIR / f"{doc_id}.json", result)

        if result["ok"]:
            extracted_rows = result["data"].get("rows", [])
            extraction_rows = build_extraction_rows(doc_id, expected_rows, extracted_rows)
            print("vote extraction rows:", len(extracted_rows))

    # เลือกผลสุดท้าย
    final_rows, strategy = choose_final_rows(rule_rows, extraction_rows)

    document_result = {
        "doc_id": doc_id,
        "pages": [p.name for p in pages],
        "strategy_used": strategy,
        "rows": final_rows,
    }

    write_json(PARSED_DIR / f"{doc_id}.json", document_result)
    return document_result


def export_submission(template, results):
    vote_lookup = {}
    for doc in results.values():
        for row in doc["rows"]:
            vote_lookup[row["id"]] = row["votes"]

    # filled
    output_rows = []
    for row in template["raw_rows"]:
        new_row = dict(row)
        row_id = str(new_row[template["id_column"]]).strip()
        if row_id in vote_lookup:
            new_row[template["votes_column"]] = vote_lookup[row_id]
        output_rows.append(new_row)

    with SUBMISSION_FILLED_PATH.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=template["fieldnames"])
        writer.writeheader()
        writer.writerows(output_rows)

    # final
    with SUBMISSION_FINAL_PATH.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "votes"])
        writer.writeheader()
        for row in output_rows:
            writer.writerow({
                "id": row[template["id_column"]],
                "votes": row[template["votes_column"]],
            })

    print("saved ->", SUBMISSION_FILLED_PATH)
    print("saved ->", SUBMISSION_FINAL_PATH)


def run_pipeline(max_docs=None, rerun_docs=None):
    checkpoint = load_checkpoint(CHECKPOINT_PATH)
    done_docs = set(checkpoint["done_docs"])
    rerun_docs = set(rerun_docs or [])

    docs = list(doc_groups.items())
    if max_docs is not None:
        docs = docs[:max_docs]

    print("resume:", len(done_docs), "/", len(doc_groups))

    for i, (doc_id, pages) in enumerate(docs, 1):
        if doc_id in done_docs and doc_id not in rerun_docs:
            print(f"[skip {i}/{len(docs)}] {doc_id}")
            continue

        if doc_id not in template["by_doc"]:
            print(f"[skip {i}/{len(docs)}] {doc_id} missing from template")
            continue

        result = process_document(doc_id, pages, template["by_doc"][doc_id])
        checkpoint["results"][doc_id] = result
        checkpoint["strategy_used"][doc_id] = result["strategy_used"]

        done_docs.add(doc_id)
        checkpoint["done_docs"] = sorted(done_docs)
        save_checkpoint(CHECKPOINT_PATH, checkpoint)

        print(f"[saved checkpoint] {doc_id} ({i}/{len(docs)})")

    export_submission(template, checkpoint["results"])
    return checkpoint

## 📄 ทดลองรัน pipeline กับเอกสารจำนวน 3 docs เพื่อเช็กว่า OCR, mapping, และการสร้างไฟล์ submission ทำงานได้ถูกต้องก่อนรันทั้งชุด

In [25]:
checkpoint = run_pipeline(max_docs=3)

resume: 0 / 300
[doc] constituency_10_1 pages=1
page 1/1 -> constituency_10_1_page2.png
rows extracted: 18
vote extraction rows: 18
[saved checkpoint] constituency_10_1 (1/3)
[doc] constituency_10_10 pages=1
page 1/1 -> constituency_10_10_page2.png
rows extracted: 16
vote extraction rows: 16
[saved checkpoint] constituency_10_10 (2/3)
[doc] constituency_10_11 pages=2
page 1/2 -> constituency_10_11.png
rows extracted: 1
page 2/2 -> constituency_10_11_page2.png
rows extracted: 17
vote extraction rows: 18
[saved checkpoint] constituency_10_11 (3/3)
saved -> /content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/submission_filled.csv
saved -> /content/drive/MyDrive/SuperAI_SS6/Hackathon_Week2/data/submission_final.csv


## 📄 รัน pipeline กับเอกสารทั้งหมดในชุดข้อมูล เพื่อสร้างผลลัพธ์ final สำหรับ export เป็นไฟล์ submission พร้อมส่งแข่งขัน

In [26]:
checkpoint = run_pipeline()

resume: 3 / 300
[skip 1/300] constituency_10_1
[skip 2/300] constituency_10_10
[skip 3/300] constituency_10_11
[doc] constituency_10_12 pages=1
page 1/1 -> constituency_10_12_page2.png
rows extracted: 16
vote extraction rows: 16
[saved checkpoint] constituency_10_12 (4/300)
[doc] constituency_10_13 pages=1
page 1/1 -> constituency_10_13_page2.png
rows extracted: 15
vote extraction rows: 15
[saved checkpoint] constituency_10_13 (5/300)
[doc] constituency_10_14 pages=1
page 1/1 -> constituency_10_14_page2.png
rows extracted: 16
vote extraction rows: 16
[saved checkpoint] constituency_10_14 (6/300)
[doc] constituency_10_16 pages=1
page 1/1 -> constituency_10_16_page2.png
rows extracted: 15
vote extraction rows: 15
[saved checkpoint] constituency_10_16 (7/300)
[doc] constituency_10_17 pages=1
page 1/1 -> constituency_10_17_page2.png
rows extracted: 15
vote extraction rows: 15
[saved checkpoint] constituency_10_17 (8/300)
[doc] constituency_10_18 pages=1
page 1/1 -> constituency_10_18_page2